# **CODE**

# 🕵️‍♂️ Misi 1: Investigasi Total Omzet (Revenue Drop Check)

In [14]:
# 1. HIPOTESIS 1: VOLUME DROP (Active Users)
q_hipotesis_1 = """
SELECT
    CASE
        WHEN MONTH(CAST(order_time AS TIMESTAMP)) IN (4, 5, 6) THEN 'Q2 (Apr-Jun)'
        ELSE 'Q3 (Jul-Sep)'
    END AS quarter,
    COUNT(DISTINCT user_id) AS active_users,
    COUNT(order_id) AS total_order_intents
FROM orders
GROUP BY 1;
"""

# 🕵️‍♂️ Misi 2: BEHAVIOR DROP (Fulfillment & Wait Time)

In [15]:
# 2. HIPOTESIS 2: BEHAVIOR DROP (Fulfillment & Wait Time)
q_hipotesis_2 = """
SELECT
    CASE
        WHEN MONTH(CAST(order_time AS TIMESTAMP)) IN (4, 5, 6) THEN 'Q2 (Apr-Jun)'
        ELSE 'Q3 (Jul-Sep)'
    END AS quarter,
    ROUND(100.0 * COUNT(CASE WHEN order_status = 'Completed' THEN order_id END) / COUNT(order_id), 2) AS fulfillment_rate_pct,
    ROUND(100.0 * COUNT(CASE WHEN order_status = 'Cancelled_Driver' THEN order_id END) / COUNT(order_id), 2) AS driver_cancel_rate_pct,
    ROUND(AVG(pickup_wait_time_min), 1) AS avg_wait_time_min
FROM orders
GROUP BY 1;
"""

# 🕵️‍♂️ Misi 3: BASKET SIZE DROP (Promo Dependency)

In [16]:
q_hipotesis_3 = """
SELECT
    CASE
        WHEN MONTH(CAST(order_time AS TIMESTAMP)) IN (4, 5, 6) THEN 'Q2 (Apr-Jun)'
        ELSE 'Q3 (Jul-Sep)'
    END AS quarter,
    ROUND(100.0 * AVG(is_promo_used), 2) AS promo_usage_pct,
    ROUND(SUM(discount_amount), 0) AS total_discount_budget,
    ROUND(AVG(gross_fare), 0) AS avg_gross_fare
FROM orders
GROUP BY 1;
"""

# 🕵️‍♂️ Result

In [18]:
from IPython.display import display, HTML

# 1. Eksekusi query menjadi DataFrame
df_res1 = con.execute(q_hipotesis_1).df()
df_res2 = con.execute(q_hipotesis_2).df()
df_res3 = con.execute(q_hipotesis_3).df()

# 2. Fungsi untuk mempercantik tampilan tabel di Colab
def style_df(df, title):
    display(HTML(f"<h3 style='color: #1a365d; font-family: Arial; border-bottom: 2px solid #3182ce; padding-bottom: 4px;'>{title}</h3>"))
    styled = (df.style
              .set_table_styles([
                  {'selector': 'th', 'props': [('background-color', '#3182ce'), ('color', 'white'), ('font-family', 'Arial'), ('text-align', 'center'), ('padding', '10px'), ('font-weight', 'bold')]},
                  {'selector': 'td', 'props': [('font-family', 'Arial'), ('text-align', 'center'), ('padding', '8px')]},
                  {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#ebf8ff')]}
              ])
              .set_properties(**{'border': '1px solid #cbd5e0'})
            )
    display(styled)

# Result dengan UI Intraktif
style_df(df_res1, "📊 Misi 1: Active Users & Order Intents")
style_df(df_res2, "📊 Misi 2: Operational Performance (Fulfillment & Wait Time)")
style_df(df_res3, "📊 Misi 3: Promo & Pricing Dependency")

# 3. Simpan ke File CSV Terpisah
df_res1.to_csv('result_active_users.csv', index=False)
df_res2.to_csv('result_operational_performance.csv', index=False)
df_res3.to_csv('result_promo_pricing.csv', index=False)

# 4. Simpan ke File Excel (.xlsx) dengan Multiple Sheets
with pd.ExcelWriter('q3_investigation_results.xlsx', engine='openpyxl') as writer:
    df_res1.to_excel(writer, sheet_name='Active_Users', index=False)
    df_res2.to_excel(writer, sheet_name='Operational_Perf', index=False)
    df_res3.to_excel(writer, sheet_name='Promo_Pricing', index=False)

print("\n✅ Tampilan berhasil dirapikan, serta file CSV dan 'q3_investigation_results.xlsx' berhasil disimpan!")

,quarter,active_users,total_order_intents
0,Q2 (Apr-Jun),1491,7459
1,Q3 (Jul-Sep),1486,7541


,quarter,fulfillment_rate_pct,driver_cancel_rate_pct,avg_wait_time_min
0,Q2 (Apr-Jun),82.410000,4.560000,4.500000
1,Q3 (Jul-Sep),58.530000,16.280000,16.000000


,quarter,promo_usage_pct,total_discount_budget,avg_gross_fare
0,Q2 (Apr-Jun),54.580000,45902000.000000,37197.000000
1,Q3 (Jul-Sep),18.060000,15056000.000000,37164.000000



✅ Tampilan berhasil dirapikan, serta file CSV dan 'q3_investigation_results.xlsx' berhasil disimpan!
